<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Developing Applications with Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">2. Grouping and Aggregating Data</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 2.2 Demo - Grouping and Aggregating Data
### Duration - 18 Minutes

This demonstration explores grouping and aggregation using the NYC taxi trips sample dataset: basic grouping, combining multiple aggregations, window functions, and the SQL equivalents.

## Learning Objectives

**By the end of this demo, you will be able to:**

- Group data by one or more columns and count, sum, and average within groups
- Combine multiple aggregations in a single `agg()` call with aliased results
- Apply window functions to rank and bucket aggregated results
- Express the same aggregations in Spark SQL

## REQUIRED - SELECT CLASSIC COMPUTE

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select your <strong>Classic Compute</strong> cluster from the compute drop-down at the top-right of the notebook. Be aware that <strong>Serverless</strong> is selected by default.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li>If your cluster is not shown in the drop-down, select <strong>More</strong>, then choose your cluster in the <strong>Attach to an existing compute resource</strong> dialog.</li>
<li>If your cluster has terminated, open <strong>Compute</strong> in the left navigation, start your cluster, then return here and attach it.</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This course was developed and tested on <strong>Databricks Runtime 17.3 LTS (Apache Spark 4.0)</strong>. Several demonstrations reference the Spark UI and query execution plans, which are not exposed on Serverless compute.</p>
</div>
</div>
</div>

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It validates that the **01_Demo_Required_Setup** notebook has been run, sets your default catalog, and sets the default schema to `spark_dev`.

In [0]:
%run ../Includes/Classroom-Setup-2

## B. Data Setup and Loading

We will use the NYC taxi trips table from the Databricks `samples` catalog: one row per taxi trip with pickup and dropoff times, distance, fare, and zip codes.

In [0]:
%python
# Read the NYC Taxi Trips Table
trips_df = spark.read.table("samples.nyctaxi.trips")

In [0]:
%python
# Print the Schema
trips_df.printSchema()

In [0]:
%python
# Visually Inspect a Subset of the Data
display(trips_df.limit(10))

## C. Basic Grouping Operations

Let's start with the simplest possible question: which pickup zip codes have the most trips?

In [0]:
%python
# Count Trips per Pickup Zip
from pyspark.sql.functions import desc

trips_by_zip_df = (
    trips_df
    .groupBy("pickup_zip")
    .count()
    .orderBy(desc("count"))
)

display(trips_by_zip_df.limit(10))

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">The <code>groupBy()</code> call returns a <code>GroupedData</code> object - it is the chained <code>count()</code> that defines the aggregation, and nothing executes until <code>display()</code> (an action) triggers the job. Check the Spark UI for the job this cell ran: the shuffle that redistributed rows by <code>pickup_zip</code> appears as an exchange between two stages.</p>
</div>

## D. Combining Multiple Aggregations

One pass over the data can compute several aggregates per group. Use `agg()` with aliased aggregation functions.

In [0]:
%python
# Aggregate Distance and Fares per Pickup Zip
from pyspark.sql.functions import avg, sum, count, round, desc

zip_summary_df = (
    trips_df
    .groupBy("pickup_zip")
    .agg(
        count("*").alias("trip_count"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(sum("fare_amount"), 2).alias("total_fares"),
        round(avg("fare_amount"), 2).alias("avg_fare")
    )
    .orderBy(desc("total_fares"))
)

display(zip_summary_df.limit(10))

In [0]:
%python
# Aggregate with Dictionary Syntax
# The dictionary form is compact but does not support aliasing
display(
    trips_df
    .groupBy("pickup_zip")
    .agg({"trip_distance": "avg", "fare_amount": "sum"})
    .limit(10)
)

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">The dictionary form generates column names like <code>avg(trip_distance)</code>. For results you intend to reference downstream, prefer the explicit function form with <code>alias()</code>.</p>
</div>

## E. Window Functions

Window functions let us rank and bucket the aggregated results without collapsing them further. Let's rank pickup zips by total fares and split them into quintiles.

In [0]:
%python
# Rank Pickup Zips by Total Fares
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, ntile, desc

# Window specification: order all rows by total fares, highest first
fares_window = Window.orderBy(desc("total_fares"))

ranked_zips_df = (
    zip_summary_df
    .withColumn("revenue_rank", rank().over(fares_window))
    .withColumn("revenue_quintile", ntile(5).over(fares_window))
)

display(ranked_zips_df.limit(20))

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">Every input row is preserved - the rank and quintile are attached as new columns. A <code>groupBy()</code> would have collapsed the rows instead. Also note the warning Spark logs about a window with no <code>partitionBy</code>: all rows must pass through a single partition to be ranked globally, which is fine here because the aggregated data is small.</p>
</div>

In [0]:
%python
# Compare Each Zip to its Quintile Average
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, round, col

# Window partitioned by quintile: each row sees the average of its bucket
quintile_window = Window.partitionBy("revenue_quintile")

display(
    ranked_zips_df
    .withColumn("quintile_avg_fares", round(avg("total_fares").over(quintile_window), 2))
    .withColumn("vs_quintile_avg", round(col("total_fares") - col("quintile_avg_fares"), 2))
    .select("pickup_zip", "total_fares", "revenue_quintile", "quintile_avg_fares", "vs_quintile_avg")
    .limit(20)
)

## F. The Same Aggregations in Spark SQL

Every aggregation above has a direct SQL equivalent. Register the DataFrame as a temporary view and express the summary in SQL.

In [0]:
%python
# Register a Temp View
trips_df.createOrReplaceTempView("trips")

In [0]:
-- Aggregate per Pickup Zip with SQL
SELECT
  pickup_zip,
  COUNT(*) AS trip_count,
  ROUND(AVG(trip_distance), 2) AS avg_distance,
  ROUND(SUM(fare_amount), 2) AS total_fares,
  ROUND(MIN(fare_amount), 2) AS min_fare,
  ROUND(MAX(fare_amount), 2) AS max_fare
FROM trips
GROUP BY pickup_zip
ORDER BY total_fares DESC
LIMIT 10;

In [0]:
-- Rank with a SQL Window Function
SELECT
  pickup_zip,
  ROUND(SUM(fare_amount), 2) AS total_fares,
  RANK() OVER (ORDER BY SUM(fare_amount) DESC) AS revenue_rank
FROM trips
GROUP BY pickup_zip
ORDER BY revenue_rank
LIMIT 10;

## Key Takeaways

**Grouping fundamentals:**

- `groupBy()` shuffles rows by key and aggregates each group in parallel; it is lazy until an action runs
- Use `agg()` with aliased aggregation functions to compute several metrics in one pass

**Window functions:**

- Rank, bucket, and compare rows while keeping row-level detail
- A window without `partitionBy` funnels all rows through one partition - acceptable for small aggregated data, dangerous on raw data

**SQL parity:**

- Every DataFrame aggregation has a direct `GROUP BY` or `OVER` equivalent in Spark SQL, executed by the same engine

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>